In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1LsdO7VYeHkTyx6EZe38rvtTPxwKI3Dbg

Retrieving folder contents
Retrieving folder 1jwboRpJIFurytqWMhSjlAs0EhcnAFc8m 01_metaspace_codepoint_bert
Processing file 1x_i8T28fZ2SRZoDc0WvivQ8S9nER-75t metrics.json
Processing file 16kOGbjrtJ39qaR8e5-dKgdQIZgEL17xY tokenizer.json
Retrieving folder 1B-bXZOh6CDIqyaYdxQYFA3wnxVs6N0_v 01_metaspace_codepoint_bpe
Processing file 1NI8096yKiidv6a8AJKbCI2-fcYiyzbYq metrics.json
Processing file 1Iuc6wcJGkqugOwjGXX0C9lwPDSK9mfXM tokenizer.json
Retrieving folder 12FOO-m3VromXXGyk9nYkrrTb0yIiz-tl 01_metaspace_codepoint_unigram
Processing file 1HbL_Mo4EF7BUHKAu4xTpVdDSQJqAreEy metrics.json
Processing file 1dmWWgcrtC0vIN749kHVkItF3EUrfbsl6 tokenizer.json
Retrieving folder 1TAWhLjEIflpTegU-KQlOizUzaqw--Gty 02_metaspace_grapheme_bert
Processing file 1IrDde5BT1biWJRKi0mkf2J-HwUm-ieOH metrics.json
Processing file 1Jtx9Osxh_s6CMlnRGToNUbCcJvZy_-eW tokenizer.json
Retrieving folder 1QymWZ7AIhiDc8WcPtYhrt0ugZ9IQto_b 02_metaspace_grapheme_bpe
Processing file 1uFpb3iIKULs6zkQ6Lif1pAaCHxot2WrS metrics.json

-------

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
!gdown 1on6htYchGSzTdWwoEUXjg2IvT15oj_vX

Downloading...
From (original): https://drive.google.com/uc?id=1on6htYchGSzTdWwoEUXjg2IvT15oj_vX
From (redirected): https://drive.google.com/uc?id=1on6htYchGSzTdWwoEUXjg2IvT15oj_vX&confirm=t&uuid=ac927bec-be77-4827-8a9e-e52ce9e4a1dc
To: /content/test.txt
100% 921M/921M [00:10<00:00, 89.6MB/s]


In [ ]:
!gdown 1dxCUqWeboST_uN_ouo2D_WWoLYn6z0bm

Downloading...
From (original): https://drive.google.com/uc?id=1dxCUqWeboST_uN_ouo2D_WWoLYn6z0bm
From (redirected): https://drive.google.com/uc?id=1dxCUqWeboST_uN_ouo2D_WWoLYn6z0bm&confirm=t&uuid=5d9ecfd2-429f-450d-9e4c-c7b20eaac07e
To: /content/test_sandhi_marked.txt
100% 930M/930M [00:09<00:00, 102MB/s]


In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file('tokenizers/01_metaspace_codepoint_bpe/tokenizer.json')

In [ ]:
import transformers
from transformers import RobertaConfig
from transformers import RobertaForMaskedLM

In [ ]:
config = RobertaConfig(
    vocab_size=len(tokenizer.get_vocab()),
    max_position_embeddings=512+3+1,
    num_attention_heads=12,
    num_hidden_layers=6,
    type_vocab_size=1,
    pad_token_id=tokenizer.token_to_id('[PAD]'),
    bos_token_id=tokenizer.token_to_id('[CLS]'),
    eos_token_id=tokenizer.token_to_id('[SEP]'),
    add_cross_attention=False
)

model = RobertaForMaskedLM(config=config)
print('Number of paramerters', model.num_parameters())

Number of paramerters 68125952


In [ ]:
from transformers import PreTrainedTokenizerFast

raw_tokenizer = Tokenizer.from_file('tokenizers/01_metaspace_codepoint_bpe/tokenizer.json')

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
     unk_token="[UNK]",
    pad_token="[PAD]",
    mask_token="[MASK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
)

In [ ]:
print("config.vocab_size:", config.vocab_size)
print("len(tokenizer):", len(tokenizer))
print("mask/pad/cls/sep/unk ids:",
      tokenizer.mask_token_id, tokenizer.pad_token_id,
      tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.unk_token_id)

config.vocab_size: 32000
len(tokenizer): 32000
mask/pad/cls/sep/unk ids: 4 3 1 2 0


## Build custom DatasetClass from PyTorch

In [ ]:
import torch
torch.cuda.init()
torch.zeros(1).cuda()   # if this alone throws, it's a driver/session issue, not your code

tensor([0.], device='cuda:0')

In [ ]:
from datasets import load_dataset

data_files = {'test' : 'test.txt',
              'train' : 'test_sandhi_marked.txt'}

data = load_dataset('text', data_files=data_files)

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
data

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 3409318
    })
    train: Dataset({
        features: ['text'],
        num_rows: 3409318
    })
})

In [ ]:
data['train'][124587]

{'text': 'எவ்வாறாயினும் , அற்புதப்பதிகம் , மூத்தபதிகம் என்⟂ற சிறப்பைப் பெற்றிருப்பதும் , பதினோராம் திருமறையில் இடம் பெற்றிருப்பதும் குறிப்பிடத்தக்கதாகும்'}

### Build trainer

#### Wrap tokenizer Tokenizer into a PreTokenizerFast object from transformers

In [ ]:
from transformers import PreTrainedTokenizerFast

raw_tokenizer = Tokenizer.from_file('tokenizers/01_metaspace_codepoint_bpe/tokenizer.json')

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
     unk_token="[UNK]",
    pad_token="[PAD]",
    mask_token="[MASK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='output',
    eval_strategy='epoch',
    num_train_epochs=1,
    learning_rate=1e-4,
    weight_decay=0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=True,
    save_total_limit=1
)

In [ ]:
raw_tokenizer.enable_truncation(max_length=512)

def tokenize_function(examples):
  encodings = raw_tokenizer.encode_batch(examples['text'])
  return {
      'input_ids' : [e.ids for e in encodings],
      'attention_mask' : [e.attention_mask for e in encodings],
  }

tokenized_data = data.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
)

Map:   0%|          | 0/3409318 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409318 [00:00<?, ? examples/s]

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['test']
)

trainer.train()

Epoch,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 